# LeetCode 638 — Shopping Offers bằng Python

Link đề: https://leetcode.com/problems/shopping-offers/ · Độ khó: **Medium** · Chủ đề: Array, Dynamic Programming, Memoization, Backtracking, Bitmask (nén trạng thái), Unbounded Knapsack

Skeleton LeetCode:

```python
class Solution:
    def shoppingOffers(self, price: List[int], special: List[List[int]], needs: List[int]) -> int:
        ...
```

Ghi chú: ví dụ 1-2 lấy từ đề; các ví dụ khác do notebook tự đặt (đáp án đã kiểm chứng bằng code độc lập), cần thì đối chiếu lại đề gốc.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: có tối đa 6 mặt hàng, mỗi mặt hàng có giá lẻ; có nhiều **gói ưu đãi** (combo), mỗi gói gồm vài món với một giá trọn gói; phải mua **đúng** số lượng cần (không mua dư) với tổng tiền **nhỏ nhất**; mỗi gói dùng được **nhiều lần**.
2. Nắm các công cụ Python cần dùng: `tuple` (khóa của từ điển), `all`/`any`/`zip`, list comprehension, `functools.lru_cache` (ghi nhớ kết quả), quay lui "chọn - thử - hoàn tác", mã hóa trạng thái thành một số nguyên (cơ số 11), `itertools.product`.
3. Đi từ **vét cạn đệ quy không nhớ** -> **đệ quy có nhớ theo trạng thái "còn cần mua gì"** -> **quay lui theo chỉ số gói (tránh đếm trùng thứ tự) kèm cắt tỉa** -> **quy hoạch động từ dưới lên (knapsack nhiều chiều)**.
4. Hiểu hai mấu chốt: (a) "trạng thái" của bài là vector số lượng **còn cần mua**; có tối đa `11^6` trạng thái nên **ghi nhớ** là đủ; (b) mỗi bước chỉ có hai lựa chọn: **dùng một gói** (nếu không vượt nhu cầu) hoặc **mua lẻ phần còn lại**; thứ tự dùng gói không quan trọng.
5. Tránh các lỗi kinh điển: cho phép mua dư, tham lam theo gói "giảm nhiều nhất", quên lựa chọn mua lẻ, gói rỗng gây đệ quy vô hạn, và dùng `list` làm khóa ghi nhớ.

## 2. Đề bài & ví dụ

Cửa hàng LeetCode có `n` mặt hàng. Mỗi mặt hàng có giá lẻ. Ngoài ra có vài **ưu đãi đặc biệt**: một ưu đãi gồm một hoặc nhiều loại hàng, với một **giá trọn gói**.

Cho:

- `price`: `price[i]` là giá lẻ của mặt hàng `i`.
- `needs`: `needs[i]` là số cái mặt hàng `i` bạn muốn mua.
- `special`: `special[i]` có độ dài `n + 1`; `special[i][j]` (với `j < n`) là số cái mặt hàng `j` trong ưu đãi `i`, và `special[i][n]` (phần tử cuối) là **giá của ưu đãi**.

Trả về **số tiền nhỏ nhất** phải trả để mua **đúng** các món theo `needs`, tận dụng ưu đãi tối ưu. **Không được mua dư** món nào, kể cả khi mua dư làm giảm tổng tiền. Mỗi ưu đãi dùng được **bao nhiêu lần cũng được**.

**Ví dụ 1**

- Input: `price = [2,5]`, `special = [[3,0,5],[1,2,10]]`, `needs = [3,2]`
- Output: `14`
- Giải thích: hàng A giá 2, hàng B giá 5. Ưu đãi 1: 5 đồng cho 3A. Ưu đãi 2: 10 đồng cho 1A + 2B. Cần 3A + 2B: dùng ưu đãi 2 (10 đồng cho 1A, 2B) và mua lẻ 2A (4 đồng).

**Ví dụ 2**

- Input: `price = [2,3,4]`, `special = [[1,1,0,4],[2,2,1,9]]`, `needs = [1,2,1]`
- Output: `11`
- Giải thích: dùng ưu đãi 1 (4 đồng cho 1A, 1B), rồi mua lẻ 1B (3) và 1C (4): `4 + 3 + 4 = 11`. Không được dùng ưu đãi 2 (9 đồng cho 2A, 2B, 1C) vì nó mua dư 1A.

**Ràng buộc:**

- `n == price.length == needs.length`, `1 <= n <= 6`.
- `0 <= price[i], needs[i] <= 10`.
- `1 <= special.length <= 100`, `special[i].length == n + 1`, `0 <= special[i][j] <= 50`.
- Đề đảm bảo ưu đãi có ít nhất một món khác 0 (code của ta vẫn chịu được ưu đãi rỗng cho an toàn).

In [1]:
from typing import List
from functools import lru_cache
from itertools import product
import heapq
import random
import time
import sys

examples = [
    (([2, 5], [[3, 0, 5], [1, 2, 10]], [3, 2]), 14),
    (([2, 3, 4], [[1, 1, 0, 4], [2, 2, 1, 9]], [1, 2, 1]), 11),
]
for (price, special, needs), expected in examples:
    n = len(price)
    base = sum(p * c for p, c in zip(price, needs))
    print(f"price={price} needs={needs} | mua lẻ hết = {base} | kỳ vọng = {expected}")

price=[2, 5] needs=[3, 2] | mua lẻ hết = 16 | kỳ vọng = 14
price=[2, 3, 4] needs=[1, 2, 1] | mua lẻ hết = 12 | kỳ vọng = 11


## 3. Kiến thức nền

### 3.1. Cấu trúc đầu vào: mỗi gói là "n món + giá cuối"

Mỗi phần tử `special[i]` dài `n + 1`: `n` số đầu là số lượng từng món, số **cuối** là giá. Tách bằng cắt lát: `offer[:n]` (các món) và `offer[n]` (giá). Giá **mua lẻ** các món của gói là `sum(p * c for ...)`: nếu giá gói không **rẻ hơn** thế thì gói vô dụng.

In [2]:
price = [2, 5]
special = [[3, 0, 5], [1, 2, 10]]
n = len(price)
for offer in special:
    items, offer_price = offer[:n], offer[n]
    single_cost = sum(p * c for p, c in zip(price, items))
    print(f"gói {offer}: món = {items}, giá gói = {offer_price}, mua lẻ các món = {single_cost}, "
          f"{'RẺ HƠN' if offer_price < single_cost else 'không rẻ hơn'}")

gói [3, 0, 5]: món = [3, 0], giá gói = 5, mua lẻ các món = 6, RẺ HƠN
gói [1, 2, 10]: món = [1, 2], giá gói = 10, mua lẻ các món = 12, RẺ HƠN


### 3.2. `tuple` và khóa của từ điển

`list` **thay đổi được** (mutable) nên **không** dùng làm khóa từ điển hay khóa của `lru_cache`. `tuple` bất biến, dùng được. Để ghi nhớ "trạng thái còn cần mua" ta đổi list thành tuple.

In [3]:
cache = {}
cache[(3, 2)] = 14
print(cache[(3, 2)], (3, 2) in cache, (2, 3) in cache)
try:
    cache[[3, 2]] = 14
except TypeError as e:
    print("list làm khóa ->", type(e).__name__, ":", e)
print(tuple([3, 2]) == (3, 2))

14 True False
list làm khóa -> TypeError : unhashable type: 'list'
True


### 3.3. `all`, `any`, `zip` và list comprehension

- `all(iterable)`: `True` nếu **mọi** phần tử đều đúng; `any(iterable)`: `True` nếu **có** phần tử đúng.
- `zip(a, b)` ghép từng cặp phần tử cùng vị trí.
- Điều kiện "gói `offer` dùng được khi còn cần `cur`" = **mọi** món đều `offer[j] <= cur[j]`.
- Trừ hai vector: `[c - o for c, o in zip(cur, items)]`.

In [4]:
cur = (3, 2)
for items in [(3, 0), (1, 2), (4, 0), (0, 0)]:
    fits = all(o <= c for o, c in zip(items, cur))
    nonempty = any(items)
    after = tuple(c - o for c, o in zip(cur, items))
    print(f"gói món {items}: vừa nhu cầu {cur}? {fits} | không rỗng? {nonempty} | còn lại {after if fits else '-'}")

gói món (3, 0): vừa nhu cầu (3, 2)? True | không rỗng? True | còn lại (0, 2)
gói món (1, 2): vừa nhu cầu (3, 2)? True | không rỗng? True | còn lại (2, 0)
gói món (4, 0): vừa nhu cầu (3, 2)? False | không rỗng? True | còn lại -
gói món (0, 0): vừa nhu cầu (3, 2)? True | không rỗng? False | còn lại (3, 2)


### 3.4. Ghi nhớ kết quả (memoization) với `lru_cache`

Khi một hàm đệ quy gọi lại **cùng đối số** nhiều lần, ta lưu kết quả lần đầu để lần sau tra ngay (`lru_cache`). Ví dụ kinh điển: Fibonacci.

In [5]:
calls = 0
def fib_slow(k):
    global calls
    calls += 1
    return k if k < 2 else fib_slow(k - 1) + fib_slow(k - 2)

print(fib_slow(20), "số lần gọi:", calls)

calls = 0
@lru_cache(None)
def fib_fast(k):
    global calls
    calls += 1
    return k if k < 2 else fib_fast(k - 1) + fib_fast(k - 2)

print(fib_fast(20), "số lần gọi:", calls)
print(fib_fast.cache_info())

6765 số lần gọi: 21891
6765 số lần gọi: 21
CacheInfo(hits=18, misses=21, maxsize=None, currsize=21)


Điều kiện để dùng `lru_cache`: đối số phải **hashable** (số, chuỗi, tuple) và kết quả chỉ phụ thuộc đối số.

### 3.5. Quay lui: chọn - thử - hoàn tác

Khi dùng **một danh sách chung** làm trạng thái, ta sửa nó khi "chọn", đệ quy, rồi **hoàn tác** để nhánh khác dùng lại. Quên hoàn tác = lỗi kinh điển.

In [6]:
def subsets_demo(items):
    path, out = [], []
    def dfs(i):
        if i == len(items):
            out.append(path[:])          # chụp bản sao tại thời điểm này
            return
        dfs(i + 1)                       # không chọn
        path.append(items[i])            # chọn
        dfs(i + 1)
        path.pop()                       # hoàn tác
    dfs(0)
    return out

print(subsets_demo([1, 2, 3]))

[[], [3], [2], [2, 3], [1], [1, 3], [1, 2], [1, 2, 3]]


### 3.6. Mã hóa trạng thái thành một số nguyên (cơ số 11)

Mỗi `needs[j]` nằm trong `0..10` tức **11** giá trị, nên cả vector `n` chiều có thể gói thành **một số** theo cơ số 11 (như viết một số có `n` chữ số cơ số 11). Đây là "nén trạng thái". Ưu điểm: khóa là một `int`, trừ gói bằng một phép trừ (miễn là gói vừa nhu cầu thì không bị "mượn" giữa các chữ số).

In [7]:
def encode(vec, base=11):
    code_ = 0
    for v in vec:
        code_ = code_ * base + v
    return code_

def decode(code_, n, base=11):
    out = []
    for _ in range(n):
        code_, r = divmod(code_, base)
        out.append(r)
    return out[::-1]

cur, offer = (3, 2, 1), (1, 2, 0)
a, b = encode(cur), encode(offer)
print(a, b, "hiệu mã =", a - b, "-> giải mã:", decode(a - b, 3), "| trừ vector:", [c - o for c, o in zip(cur, offer)])
print(decode(encode((10, 0, 7)), 3))

386 143 hiệu mã = 243 -> giải mã: [2, 0, 1] | trừ vector: [2, 0, 1]
[10, 0, 7]


### 3.7. `itertools.product`: duyệt mọi trạng thái theo thứ tự từ điển

`product(range(a), range(b))` duyệt `(0,0), (0,1), ..., (a-1, b-1)` theo thứ tự từ điển. Nếu hai trạng thái `s'` và `s` thỏa `s' <= s` từng thành phần và `s' != s` thì `s'` đứng **trước** `s`: dùng được cho quy hoạch động từ dưới lên.

In [8]:
states = list(product(range(4), range(3)))      # needs = [3, 2]
print(len(states), "trạng thái:", states)

12 trạng thái: [(0, 0), (0, 1), (0, 2), (1, 0), (1, 1), (1, 2), (2, 0), (2, 1), (2, 2), (3, 0), (3, 1), (3, 2)]


### 3.8. Knapsack không giới hạn (unbounded knapsack) một chiều

Bài đổi tiền: cho các gói giá, tìm tổng tiền nhỏ nhất để mua đúng `k` món, mỗi gói dùng nhiều lần. Công thức: `dp[k] = min(dp[k - size] + cost)`. Bài của ta là **phiên bản nhiều chiều** của đúng ý đó: `dp[vector còn cần] = min(dp[vector - gói] + giá gói)`.

In [9]:
def min_cost_exact(k, packs, single_price):
    """packs = [(số món, giá gói)]; có thể mua lẻ với giá single_price/món; cần đúng k món."""
    INF = float("inf")
    dp = [0] + [INF] * k
    for t in range(1, k + 1):
        dp[t] = dp[t - 1] + single_price            # mua lẻ món cuối
        for size, cost in packs:
            if size <= t:
                dp[t] = min(dp[t], dp[t - size] + cost)
    return dp[k]

print(min_cost_exact(7, [(3, 5), (5, 8)], 2))       # ví dụ: gói 5 món (8) + 2 món lẻ (4) = 12
print(min_cost_exact(6, [(3, 5)], 2))               # hai gói 3 món: 10

12
10


## 4. Ý tưởng nền tảng / cách nghĩ trực giác (tính tay)

Ví dụ 1: `price = [2,5]`, `special = [[3,0,5],[1,2,10]]`, `needs = [3,2]` (cần 3A và 2B).

**Mua lẻ hết:** `3*2 + 2*5 = 16`.

Thử từng gói:

- Gói 1 (`3A`, giá 5): nhu cầu còn `[0, 2]`; mua lẻ `2B = 10`. Tổng `5 + 10 = 15`.
- Gói 2 (`1A 2B`, giá 10): nhu cầu còn `[2, 0]`; mua lẻ `2A = 4`. Tổng `10 + 4 = 14`.
- Dùng cả hai gói: cần `4A, 2B` > nhu cầu `3A` -> **không được** (mua dư).

Tốt nhất `14`. Ta thấy cách nghĩ tự nhiên: **chọn một gói còn vừa nhu cầu, trừ nó khỏi nhu cầu, rồi giải bài toán nhỏ hơn**, và luôn có lựa chọn "mua lẻ phần còn lại".

Bảng nhanh (mỗi dòng là một cách chọn, "còn lại" là nhu cầu sau khi dùng gói):

| Cách | Đã dùng | Còn lại | Mua lẻ phần còn lại | Tổng |
|---|---|---|---|---|
| mua lẻ hết | - | `[3,2]` | 16 | 16 |
| gói 1 | 5 | `[0,2]` | 10 | 15 |
| gói 2 | 10 | `[2,0]` | 4 | 14 |
| gói 1 rồi gói 2 | - | không vừa | - | bỏ |

Nhận xét quan trọng: sau khi dùng gói 2, bài toán còn lại là "cần `[2,0]`", cũng có thể dùng gói khác trên `[2,0]` (ở đây không gói nào vừa). Đó là **cấu trúc con tối ưu**: đáp án cho `[3,2]` dựa trên đáp án của các nhu cầu nhỏ hơn.

## 5. Cách 1 — Brute Force / baseline: vét cạn đệ quy (không nhớ)

**Ý tưởng.** Định nghĩa `dfs(cur)` = số tiền nhỏ nhất để mua đúng vector `cur`. Với mỗi `cur`:

1. **Mua lẻ hết**: `sum(price[j] * cur[j])`.
2. Với **mỗi gói** còn vừa nhu cầu (và không rỗng): `giá gói + dfs(cur - gói)`.
3. Lấy `min` của tất cả.

Vì thứ tự dùng gói không quan trọng, nhưng cách này thử **mọi thứ tự**, các nhánh bị lặp rất nhiều (dùng gói A rồi B và dùng B rồi A dẫn tới cùng nhu cầu còn lại).

In [10]:
def shopping_brute(price, special, needs):
    n = len(price)
    def dfs(cur):
        best = sum(price[j] * cur[j] for j in range(n))          # mua lẻ phần còn lại
        for offer in special:
            if any(offer[:n]) and all(offer[j] <= cur[j] for j in range(n)):
                nxt = [cur[j] - offer[j] for j in range(n)]
                best = min(best, offer[n] + dfs(nxt))
        return best
    return dfs(list(needs))

for (args, expected) in examples:
    print(shopping_brute(*args), "| kỳ vọng", expected)

14 | kỳ vọng 14
11 | kỳ vọng 11


In [11]:
class Solution1:
    """Cách 1: vét cạn đệ quy, không nhớ."""
    def shoppingOffers(self, price: List[int], special: List[List[int]], needs: List[int]) -> int:
        n = len(price)

        def dfs(cur):
            best = sum(price[j] * cur[j] for j in range(n))
            for offer in special:
                if any(offer[:n]) and all(offer[j] <= cur[j] for j in range(n)):
                    nxt = [cur[j] - offer[j] for j in range(n)]
                    best = min(best, offer[n] + dfs(nxt))
            return best

        return dfs(list(needs))

for (args, expected) in examples:
    assert Solution1().shoppingOffers(*args) == expected
print("cách 1 qua các ví dụ")

cách 1 qua các ví dụ


**Độ phức tạp.** Mỗi lời gọi có thể rẽ tới `m` nhánh (`m` = số gói), độ sâu tối đa bằng tổng số món cần mua `S = sum(needs)` (mỗi gói bớt ít nhất 1 món) -> số lời gọi tối đa cỡ `m^S`: **hàm mũ**. Space: `O(S)` cho ngăn xếp đệ quy.

**Nút thắt.** Đo thử số lần gọi trên một ví dụ nhỏ để thấy sự lặp lại: mỗi lần ta lại tính cho cùng một nhu cầu còn lại.

In [12]:
def count_calls(price, special, needs, memo):
    n = len(price)
    calls = 0
    seen = {}
    def dfs(cur):
        nonlocal calls
        calls += 1
        key = tuple(cur)
        if memo and key in seen:
            return seen[key]
        best = sum(price[j] * cur[j] for j in range(n))
        for offer in special:
            if any(offer[:n]) and all(offer[j] <= cur[j] for j in range(n)):
                best = min(best, offer[n] + dfs([cur[j] - offer[j] for j in range(n)]))
        seen[key] = best
        return best
    return dfs(list(needs)), calls

price = [2, 3]
special = [[1, 0, 1], [0, 1, 2], [1, 1, 2], [2, 1, 3]]
for needs in ([2, 2], [3, 3], [4, 4], [5, 5]):
    r1, c1 = count_calls(price, special, needs, memo=False)
    r2, c2 = count_calls(price, special, needs, memo=True)
    print(f"needs={needs}: kết quả {r1}=={r2} | số lời gọi không nhớ = {c1:>7} | có nhớ = {c2}")

needs=[2, 2]: kết quả 4==4 | số lời gọi không nhớ =      34 | có nhớ = 19
needs=[3, 3]: kết quả 6==6 | số lời gọi không nhớ =     194 | có nhớ = 40
needs=[4, 4]: kết quả 8==8 | số lời gọi không nhớ =    1131 | có nhớ = 69
needs=[5, 5]: kết quả 10==10 | số lời gọi không nhớ =    6706 | có nhớ = 106


Chỉ có `(needs[0]+1)*(needs[1]+1)` trạng thái khác nhau (ví dụ `36` với `[5,5]`), nhưng cách không nhớ gọi **hàng chục nghìn** lần. Ta cần **nhớ** kết quả của mỗi trạng thái.

## 6. Ý tưởng thuật toán / bước đột phá

Hai nhận xét:

1. **Trạng thái rất ít**: kết quả của `dfs(cur)` **chỉ phụ thuộc** vector còn cần mua `cur`, mà `cur[j]` chỉ nhận giá trị `0..needs[j]`. Số trạng thái tối đa là `prod(needs[j] + 1) <= 11^6 = 1.771.561`. Vì vậy **ghi nhớ theo `cur`** biến hàm mũ thành "mỗi trạng thái tính một lần".
2. **Lọc gói vô dụng**: một gói có giá **không rẻ hơn** giá mua lẻ các món của nó thì có thể thay bằng mua lẻ mà không tệ hơn -> bỏ đi. Gói rỗng (không món nào) cũng bỏ (nó không tiến bộ gì và gây đệ quy vô hạn).

Tổng hợp thành quy hoạch động: `f(cur) = min( mua lẻ cur , min_{gói vừa cur} giá gói + f(cur - gói) )`, `f(0) = 0`.

Từ cùng công thức ta có ba cách thực hiện:

- Cách 2: **đệ quy có nhớ** theo `cur` (từ trên xuống).
- Cách 3: **quay lui theo chỉ số gói** (mỗi tập gói chỉ thử một lần, không thử mọi hoán vị) + cắt tỉa theo chi phí tốt nhất hiện có.
- Cách 4: **quy hoạch động từ dưới lên** duyệt mọi trạng thái từ nhỏ đến lớn.

## 7. Các cách tiếp theo

### Cách 2 — Đệ quy có nhớ theo trạng thái `cur` (khuyên dùng)

**Giải thích từng đoạn.**

1. **Lọc gói**: giữ gói nếu `any(món)` và `giá gói < giá mua lẻ các món`.
2. `dfs(cur)` với `cur` là **tuple** (để làm khóa ghi nhớ).
3. Khởi tạo `best` = mua lẻ hết `cur`.
4. Với mỗi gói còn vừa (`offer[j] <= cur[j]` với mọi `j`): `best = min(best, giá gói + dfs(cur - gói))`.
5. Gọi `dfs(tuple(needs))`.

Demo từng đoạn (bước lọc, rồi tính một trạng thái):

In [13]:
price, special, needs = [2, 3, 4], [[1, 1, 0, 4], [2, 2, 1, 9], [1, 0, 0, 5]], [1, 2, 1]
n = len(price)

# đoạn 1: lọc gói
offers = [o for o in special if any(o[:n]) and o[n] < sum(p * c for p, c in zip(price, o[:n]))]
print("gói giữ lại:", offers, "| gói [1,0,0,5] bị bỏ vì 5 >= 2 (đắt hơn mua lẻ)")

# đoạn 2-4: một lần "mở rộng" từ trạng thái (1, 2, 1)
cur = (1, 2, 1)
print("mua lẻ hết:", sum(p * c for p, c in zip(price, cur)))
for o in offers:
    if all(o[j] <= cur[j] for j in range(n)):
        print("dùng gói", o, "-> còn lại", tuple(c - x for c, x in zip(cur, o[:n])), "+ giá gói", o[n])

gói giữ lại: [[1, 1, 0, 4], [2, 2, 1, 9]] | gói [1,0,0,5] bị bỏ vì 5 >= 2 (đắt hơn mua lẻ)
mua lẻ hết: 12
dùng gói [1, 1, 0, 4] -> còn lại (0, 1, 1) + giá gói 4


**Code nộp LeetCode (cách 2) - cũng là code chính ở mục 9:**

In [14]:
class Solution2:
    """Cách 2: đệ quy có nhớ theo trạng thái còn cần mua."""
    def shoppingOffers(self, price: List[int], special: List[List[int]], needs: List[int]) -> int:
        n = len(price)
        offers = [o for o in special
                  if any(o[:n]) and o[n] < sum(price[j] * o[j] for j in range(n))]

        @lru_cache(None)
        def dfs(cur):
            best = sum(price[j] * cur[j] for j in range(n))
            for o in offers:
                if all(o[j] <= cur[j] for j in range(n)):
                    best = min(best, o[n] + dfs(tuple(cur[j] - o[j] for j in range(n))))
            return best

        return dfs(tuple(needs))

for (args, expected) in examples:
    assert Solution2().shoppingOffers(*args) == expected
print("cách 2 qua các ví dụ")

cách 2 qua các ví dụ


**Độ phức tạp (cách 2).** Số trạng thái `P = prod(needs[j] + 1) <= 11^n`. Mỗi trạng thái thử `m` gói, mỗi lần kiểm tra/trừ vector tốn `O(n)`. **Time `O(P * m * n)`**, Space `O(P * n)` cho bảng nhớ (mỗi khóa là tuple `n` phần tử) cộng ngăn xếp đệ quy độ sâu tối đa `sum(needs)`. Trong thực tế chỉ các trạng thái **đến được** từ `needs` mới được tính, thường ít hơn `P` nhiều.

### Cách 3 — Quay lui theo chỉ số gói (multiset) + cắt tỉa

**Giải thích.**

- Cách 1 và 2 thử gói theo *thứ tự*, nên "gói A rồi gói B" và "gói B rồi gói A" là hai nhánh. Nhưng một phương án thực chất là **một tập gói (có lặp)** cộng với mua lẻ phần còn lại. Ta chỉ cần liệt kê mỗi tập **một lần**: khi đã chọn gói số `k`, các gói tiếp theo chỉ được chọn có chỉ số `>= k` (được lặp lại chính `k`, vì mỗi gói dùng không giới hạn).
- Dùng một vector `cur` **chung** và quay lui: trừ gói đi, đệ quy, cộng gói lại.
- Với mỗi tập gói đã chọn (kể cả rỗng), tính tổng = (tiền các gói) + (mua lẻ phần còn lại) và cập nhật đáp án tốt nhất.
- **Cắt tỉa:** nếu tiền các gói đã chọn đã `>= best` thì dừng nhánh (chọn thêm gói chỉ tốn thêm tiền, vì giá >= 0).

Không có bảng nhớ, bộ nhớ chỉ là ngăn xếp đệ quy; nhưng số tập gói có thể lớn khi có nhiều gói nhỏ.

In [15]:
def demo_multiset(price, special, needs):
    n = len(price)
    offers = [o for o in special if any(o[:n]) and o[n] < sum(price[j] * o[j] for j in range(n))]
    cur = list(needs)
    best = sum(price[j] * cur[j] for j in range(n))
    chosen = []

    def dfs(start, cost):
        nonlocal best
        total = cost + sum(price[j] * cur[j] for j in range(n))
        if total < best:
            best = total
        for k in range(start, len(offers)):
            o = offers[k]
            if all(o[j] <= cur[j] for j in range(n)):
                for j in range(n):
                    cur[j] -= o[j]                # chọn
                chosen.append(k)
                print("  " * len(chosen), "chọn gói", k, "-> còn", cur, "| tiền gói =", cost + o[n])
                dfs(k, cost + o[n])               # k (không phải k+1): gói k dùng lại được
                chosen.pop()
                for j in range(n):
                    cur[j] += o[j]                # hoàn tác
    dfs(0, 0)
    return best

print("đáp án:", demo_multiset([2, 5], [[3, 0, 5], [1, 2, 10]], [3, 2]))

   chọn gói 0 -> còn [0, 2] | tiền gói = 5
   chọn gói 1 -> còn [2, 0] | tiền gói = 10
đáp án: 14


**Code nộp LeetCode (cách 3):**

In [16]:
class Solution3:
    """Cách 3: quay lui theo chỉ số gói (mỗi tập gói thử một lần) + cắt tỉa."""
    def shoppingOffers(self, price: List[int], special: List[List[int]], needs: List[int]) -> int:
        n = len(price)
        offers = [o for o in special
                  if any(o[:n]) and o[n] < sum(price[j] * o[j] for j in range(n))]
        cur = list(needs)
        best = sum(price[j] * cur[j] for j in range(n))

        def dfs(start, cost):
            nonlocal best
            if cost >= best:                       # cắt tỉa
                return
            best = min(best, cost + sum(price[j] * cur[j] for j in range(n)))
            for k in range(start, len(offers)):
                o = offers[k]
                if all(o[j] <= cur[j] for j in range(n)):
                    for j in range(n):
                        cur[j] -= o[j]
                    dfs(k, cost + o[n])
                    for j in range(n):
                        cur[j] += o[j]

        dfs(0, 0)
        return best

for (args, expected) in examples:
    assert Solution3().shoppingOffers(*args) == expected
print("cách 3 qua các ví dụ")

cách 3 qua các ví dụ


**Độ phức tạp (cách 3).** Số nút của cây quay lui = số **tập gói (có lặp)** có tổng không vượt `needs`, tối đa cỡ `C(S + m, m)` (với `S = sum(needs)`); mỗi nút tốn `O(n)` để cập nhật và tính phần mua lẻ. Cắt tỉa giúp bớt nhiều nhưng **không đảm bảo** trường hợp xấu; Space `O(S)`. Cách này gọn, không cần bảng nhớ, tốt khi ít gói; khi nhiều gói nhỏ thì cách 2 an toàn hơn.

### Cách 4 — Quy hoạch động từ dưới lên (knapsack nhiều chiều)

**Giải thích.**

- Bảng `dp` có `P = prod(needs[j] + 1)` ô, mỗi ô ứng với một vector `cur` (`0 <= cur[j] <= needs[j]`), **đánh số** bằng cách mã hóa kiểu "cơ số hỗn hợp": `idx = sum(cur[j] * stride[j])`, `stride[n-1] = 1`, `stride[j] = stride[j+1] * (needs[j+1] + 1)`.
- Duyệt `cur` bằng `itertools.product(range(needs[0]+1), ...)`: thứ tự này tăng dần theo `idx` và mọi `cur - gói` (nhỏ hơn từng thành phần) có `idx` **nhỏ hơn**, đã được tính.
- `dp[idx] = min( mua lẻ cur , min_{gói vừa} dp[idx - mã gói] + giá gói )`, `dp[0] = 0`.
- Đáp án `dp[idx(needs)] = dp[P - 1]`.

Demo bảng cho ví dụ 1 (`needs = [3,2]`, 12 ô):

In [17]:
def dp_table(price, special, needs):
    n = len(price)
    offers = [o for o in special if any(o[:n]) and o[n] < sum(price[j] * o[j] for j in range(n))]
    dims = [c + 1 for c in needs]
    stride = [1] * n
    for j in range(n - 2, -1, -1):
        stride[j] = stride[j + 1] * dims[j + 1]
    total = stride[0] * dims[0]
    dp = [0] * total
    for cur in product(*[range(d) for d in dims]):
        idx = sum(c * s for c, s in zip(cur, stride))
        best = sum(price[j] * cur[j] for j in range(n))
        for o in offers:
            if all(o[j] <= cur[j] for j in range(n)):
                off = sum(o[j] * stride[j] for j in range(n))
                best = min(best, dp[idx - off] + o[n])
        dp[idx] = best
    return dp, dims, stride

dp, dims, stride = dp_table([2, 5], [[3, 0, 5], [1, 2, 10]], [3, 2])
print("dims =", dims, "stride =", stride)
for a in range(dims[0]):
    print(f"cần {a}A:", [dp[a * stride[0] + b] for b in range(dims[1])], "(theo số B = 0,1,2)")
print("đáp án dp[3A,2B] =", dp[-1])

dims = [4, 3] stride = [3, 1]
cần 0A: [0, 5, 10] (theo số B = 0,1,2)
cần 1A: [2, 7, 10] (theo số B = 0,1,2)
cần 2A: [4, 9, 12] (theo số B = 0,1,2)
cần 3A: [5, 10, 14] (theo số B = 0,1,2)
đáp án dp[3A,2B] = 14


**Code nộp LeetCode (cách 4):**

In [18]:
class Solution4:
    """Cách 4: quy hoạch động từ dưới lên trên toàn bộ không gian trạng thái."""
    def shoppingOffers(self, price: List[int], special: List[List[int]], needs: List[int]) -> int:
        n = len(price)
        offers = [o for o in special
                  if any(o[:n]) and o[n] < sum(price[j] * o[j] for j in range(n))]
        dims = [c + 1 for c in needs]
        stride = [1] * n
        for j in range(n - 2, -1, -1):
            stride[j] = stride[j + 1] * dims[j + 1]
        codes = [sum(o[j] * stride[j] for j in range(n)) for o in offers]
        dp = [0] * (stride[0] * dims[0])
        for cur in product(*[range(d) for d in dims]):
            idx = sum(c * s for c, s in zip(cur, stride))
            best = sum(price[j] * cur[j] for j in range(n))
            for o, off in zip(offers, codes):
                if all(o[j] <= cur[j] for j in range(n)):
                    best = min(best, dp[idx - off] + o[n])
            dp[idx] = best
        return dp[-1]

for (args, expected) in examples:
    assert Solution4().shoppingOffers(*args) == expected
print("cách 4 qua các ví dụ")

cách 4 qua các ví dụ


**Độ phức tạp (cách 4).** **Time `O(P * m * n)`**, Space `O(P)` (một mảng số nguyên). Khác cách 2: **luôn** duyệt đủ `P` trạng thái kể cả không đến được từ `needs`; bù lại không có đệ quy (không lo giới hạn độ sâu) và hằng số nhỏ hơn mỗi trạng thái. Với `needs` toàn `10` và `n = 6` thì `P = 1.771.561`, nhân `m = 100` gói sẽ chậm trong Python, nên với dữ liệu cực đại cách 2 (chỉ trạng thái đến được + lọc gói) thường thực tế hơn.

**Tóm tắt bốn cách:** (1) vét cạn đệ quy mũ; (2) đệ quy có nhớ theo trạng thái; (3) quay lui theo chỉ số gói + cắt tỉa; (4) quy hoạch động từ dưới lên trên toàn bộ trạng thái.

## 8. TRACE từng bước

### 8.1. Trace cách 2 (đệ quy có nhớ) trên ví dụ 2

`price = [2,3,4]`, `special = [[1,1,0,4],[2,2,1,9]]`, `needs = [1,2,1]`. Hàm in mỗi lần **vào** một trạng thái (thụt lề theo độ sâu), việc dùng gói, kết quả trả về và trường hợp **trúng bảng nhớ**.

| Cột | Ý nghĩa |
|---|---|
| `vào cur` | trạng thái còn cần mua |
| `mua lẻ` | tiền nếu mua lẻ hết `cur` |
| `dùng gói` | gói vừa nhu cầu, sau khi dùng còn lại gì |
| `=> best` | giá trị nhỏ nhất của trạng thái này |

In [19]:
def trace_memo(price, special, needs):
    n = len(price)
    offers = [o for o in special if any(o[:n]) and o[n] < sum(price[j] * o[j] for j in range(n))]
    print("gói còn lại sau khi lọc:", offers)
    memo = {}

    def dfs(cur, depth):
        pad = "  " * depth
        if cur in memo:
            print(f"{pad}cur={cur}: TRÚNG BẢNG NHỚ = {memo[cur]}")
            return memo[cur]
        best = sum(price[j] * cur[j] for j in range(n))
        print(f"{pad}vào cur={cur}, mua lẻ = {best}")
        for o in offers:
            if all(o[j] <= cur[j] for j in range(n)):
                nxt = tuple(cur[j] - o[j] for j in range(n))
                print(f"{pad}  dùng gói {o}: còn {nxt}, giá gói {o[n]}")
                best = min(best, o[n] + dfs(nxt, depth + 2))
        memo[cur] = best
        print(f"{pad}=> best({cur}) = {best}")
        return best

    return dfs(tuple(needs), 0)

print("kết quả:", trace_memo([2, 3, 4], [[1, 1, 0, 4], [2, 2, 1, 9]], [1, 2, 1]))

gói còn lại sau khi lọc: [[1, 1, 0, 4], [2, 2, 1, 9]]
vào cur=(1, 2, 1), mua lẻ = 12
  dùng gói [1, 1, 0, 4]: còn (0, 1, 1), giá gói 4
    vào cur=(0, 1, 1), mua lẻ = 7
    => best((0, 1, 1)) = 7
=> best((1, 2, 1)) = 11
kết quả: 11


Quan sát: gói 2 (`[2,2,1]`) **không vừa** nhu cầu `(1,2,1)` vì cần 2A mà chỉ cần 1A -> không được dùng. Chỉ có gói 1: còn `(0,1,1)`, mua lẻ `3 + 4 = 7`, tổng `4 + 7 = 11`. Đúng đáp án.

### 8.2. Trace với nhiều gói để thấy trúng bảng nhớ

Ví dụ 1 có hai gói nhưng không bao giờ dùng được cả hai. Ta thử một ví dụ có nhiều cách tới cùng một trạng thái.

In [20]:
print("kết quả:", trace_memo([2, 3], [[1, 0, 1], [0, 1, 2], [1, 1, 2]], [2, 2]))

gói còn lại sau khi lọc: [[1, 0, 1], [0, 1, 2], [1, 1, 2]]
vào cur=(2, 2), mua lẻ = 10
  dùng gói [1, 0, 1]: còn (1, 2), giá gói 1
    vào cur=(1, 2), mua lẻ = 8
      dùng gói [1, 0, 1]: còn (0, 2), giá gói 1
        vào cur=(0, 2), mua lẻ = 6
          dùng gói [0, 1, 2]: còn (0, 1), giá gói 2
            vào cur=(0, 1), mua lẻ = 3
              dùng gói [0, 1, 2]: còn (0, 0), giá gói 2
                vào cur=(0, 0), mua lẻ = 0
                => best((0, 0)) = 0
            => best((0, 1)) = 2
        => best((0, 2)) = 4
      dùng gói [0, 1, 2]: còn (1, 1), giá gói 2
        vào cur=(1, 1), mua lẻ = 5
          dùng gói [1, 0, 1]: còn (0, 1), giá gói 1
            cur=(0, 1): TRÚNG BẢNG NHỚ = 2
          dùng gói [0, 1, 2]: còn (1, 0), giá gói 2
            vào cur=(1, 0), mua lẻ = 2
              dùng gói [1, 0, 1]: còn (0, 0), giá gói 1
                cur=(0, 0): TRÚNG BẢNG NHỚ = 0
            => best((1, 0)) = 1
          dùng gói [1, 1, 2]: còn (0, 0), giá gói 2
            c

Để ý dòng `TRÚNG BẢNG NHỚ`: các thứ tự dùng gói khác nhau tới cùng trạng thái (`(1,1)` hoặc `(0,0)` ...) chỉ được tính **một lần**.

### 8.3. Trace cách 3 (quay lui theo chỉ số gói)

Mỗi dòng: tập gói đang chọn (chỉ số, có thể lặp), phần còn cần mua, tiền gói, và tổng nếu mua lẻ phần còn lại.

In [21]:
def trace_multiset(price, special, needs):
    n = len(price)
    offers = [o for o in special if any(o[:n]) and o[n] < sum(price[j] * o[j] for j in range(n))]
    cur = list(needs)
    best = sum(price[j] * cur[j] for j in range(n))
    chosen = []

    def dfs(start, cost):
        nonlocal best
        rest = sum(price[j] * cur[j] for j in range(n))
        print(f"tập gói {chosen!s:<14} còn {cur} | tiền gói {cost:>3} + mua lẻ {rest:>3} = {cost + rest:>3}"
              f"{'  <- tốt nhất mới' if cost + rest < best else ''}")
        best = min(best, cost + rest)
        for k in range(start, len(offers)):
            o = offers[k]
            if all(o[j] <= cur[j] for j in range(n)):
                for j in range(n):
                    cur[j] -= o[j]
                chosen.append(k)
                dfs(k, cost + o[n])
                chosen.pop()
                for j in range(n):
                    cur[j] += o[j]
    dfs(0, 0)
    return best

print("đáp án:", trace_multiset([2, 3], [[1, 0, 1], [0, 1, 2], [1, 1, 2]], [2, 2]))

tập gói []             còn [2, 2] | tiền gói   0 + mua lẻ  10 =  10
tập gói [0]            còn [1, 2] | tiền gói   1 + mua lẻ   8 =   9  <- tốt nhất mới
tập gói [0, 0]         còn [0, 2] | tiền gói   2 + mua lẻ   6 =   8  <- tốt nhất mới
tập gói [0, 0, 1]      còn [0, 1] | tiền gói   4 + mua lẻ   3 =   7  <- tốt nhất mới
tập gói [0, 0, 1, 1]   còn [0, 0] | tiền gói   6 + mua lẻ   0 =   6  <- tốt nhất mới
tập gói [0, 1]         còn [1, 1] | tiền gói   3 + mua lẻ   5 =   8
tập gói [0, 1, 1]      còn [1, 0] | tiền gói   5 + mua lẻ   2 =   7
tập gói [0, 1, 2]      còn [0, 0] | tiền gói   5 + mua lẻ   0 =   5  <- tốt nhất mới
tập gói [0, 2]         còn [0, 1] | tiền gói   3 + mua lẻ   3 =   6
tập gói [1]            còn [2, 1] | tiền gói   2 + mua lẻ   7 =   9
tập gói [1, 1]         còn [2, 0] | tiền gói   4 + mua lẻ   4 =   8
tập gói [1, 2]         còn [1, 0] | tiền gói   4 + mua lẻ   2 =   6
tập gói [2]            còn [1, 1] | tiền gói   2 + mua lẻ   5 =   7
tập gói [2, 2]         còn [0, 

Mỗi **tập** gói chỉ xuất hiện một lần (ví dụ `[0, 1]` có, nhưng `[1, 0]` thì không): đó chính là lợi ích của việc chỉ cho chọn chỉ số `>= k`.

## 9. Code nộp LeetCode

Dùng **cách 2** (đệ quy có nhớ, có lọc gói vô dụng):

```python
class Solution:
    def shoppingOffers(self, price: List[int], special: List[List[int]], needs: List[int]) -> int:
        n = len(price)
        offers = [o for o in special
                  if any(o[:n]) and o[n] < sum(price[j] * o[j] for j in range(n))]

        @lru_cache(None)
        def dfs(cur):
            best = sum(price[j] * cur[j] for j in range(n))
            for o in offers:
                if all(o[j] <= cur[j] for j in range(n)):
                    best = min(best, o[n] + dfs(tuple(cur[j] - o[j] for j in range(n))))
            return best

        return dfs(tuple(needs))
```

(Trên LeetCode `List` và `lru_cache` thường đã được import sẵn; nếu không, thêm `from typing import List` và `from functools import lru_cache`.)

Chạy lại đúng đoạn code trên:

In [22]:
class Solution:
    def shoppingOffers(self, price: List[int], special: List[List[int]], needs: List[int]) -> int:
        n = len(price)
        offers = [o for o in special
                  if any(o[:n]) and o[n] < sum(price[j] * o[j] for j in range(n))]

        @lru_cache(None)
        def dfs(cur):
            best = sum(price[j] * cur[j] for j in range(n))
            for o in offers:
                if all(o[j] <= cur[j] for j in range(n)):
                    best = min(best, o[n] + dfs(tuple(cur[j] - o[j] for j in range(n))))
            return best

        return dfs(tuple(needs))

for (args, expected) in examples:
    got = Solution().shoppingOffers(*args)
    print(args, "->", got, "OK" if got == expected else "SAI")
    assert got == expected

([2, 5], [[3, 0, 5], [1, 2, 10]], [3, 2]) -> 14 OK
([2, 3, 4], [[1, 1, 0, 4], [2, 2, 1, 9]], [1, 2, 1]) -> 11 OK


## 10. Vì sao thuật toán đúng

**Cấu trúc phương án.** Một cách mua hợp lệ là: một **đa tập** các gói (mỗi gói dùng `k >= 0` lần) cộng với mua lẻ các món còn lại, sao cho tổng số món từng loại **bằng đúng** `needs` (không dư). Vì phép cộng giao hoán, ta có thể xếp các gói theo thứ tự tùy ý.

**Công thức đệ quy.** Gọi `f(cur)` là chi phí nhỏ nhất để mua đúng `cur`. Xét một phương án tối ưu cho `cur`:

- Nếu nó **không dùng gói nào**: toàn bộ là mua lẻ, chi phí `sum(price[j] * cur[j])`.
- Nếu nó **dùng ít nhất một gói** `g` (vừa `cur`, vì không được dư): phần còn lại của phương án là một phương án hợp lệ để mua `cur - g`, và nó phải tối ưu cho `cur - g` (nếu tốt hơn ta thay vào thì tổng rẻ hơn, mâu thuẫn). Chi phí `giá(g) + f(cur - g)`.

Lấy `min` trên mọi khả năng cho `f(cur) = min( mua lẻ , min_g giá(g) + f(cur - g) )`. Điều kiện dừng `f(0) = 0`. Đệ quy kết thúc vì mỗi gói **không rỗng** làm giảm ít nhất một thành phần (và không làm thành phần nào âm), nên tổng số món còn cần giảm thực sự.

**Lọc gói.** Gói có `giá >= mua lẻ các món của nó` có thể thay bằng mua lẻ các món đó mà không tăng chi phí, và vẫn mua đúng cùng một tập món. Nên bỏ chúng không làm mất phương án tối ưu. Gói rỗng không đóng góp gì (chỉ cộng thêm giá `>= 0`) nên bỏ cũng an toàn.

**Ghi nhớ.** `f(cur)` chỉ phụ thuộc `cur` (không phụ thuộc đường đi tới nó) nên lưu lại là hợp lệ.

**Cách 3** liệt kê mọi đa tập gói (mỗi tập một lần, theo chỉ số không giảm) và lấy `min(tiền gói + mua lẻ phần còn lại)`: tương đương hệ quả của cấu trúc phương án ở trên; cắt tỉa theo `cost >= best` đúng vì giá `>= 0`.

**Cách 4** chính là công thức `f` tính theo thứ tự tăng của `cur`: mọi `cur - g` đứng trước `cur`.

## 11. Phân tích Time / Space Complexity

Ký hiệu: `n <= 6` số mặt hàng, `m <= 100` số gói, `P = prod(needs[j] + 1) <= 11^6` số trạng thái, `S = sum(needs) <= 60`.

| Cách | Time | Space | Ghi chú |
|---|---|---|---|
| 1. Vét cạn đệ quy | cỡ `m^S` (hàm mũ) | `O(S)` | lặp lại cực nhiều trạng thái |
| 2. Đệ quy có nhớ | `O(P * m * n)` (chỉ trạng thái đến được) | `O(P * n)` | **dùng để nộp** |
| 3. Quay lui theo chỉ số + cắt tỉa | số đa tập gói, tối đa `C(S + m, m)` | `O(S)` | nhanh khi ít gói; không đảm bảo xấu nhất |
| 4. DP từ dưới lên | `O(P * m * n)` (mọi trạng thái) | `O(P)` | không đệ quy, hằng số nhỏ |

Trong thực tế các test của LeetCode có `needs` và số gói vừa phải nên cách 2 chạy rất nhanh. Cận `11^6 * 100 * 6` chỉ là cận trên lý thuyết.

## 12. Bộ test

### 12.1. Ví dụ của đề, ví dụ tự đặt và oracle độc lập

Oracle độc lập: **Dijkstra** trên đồ thị trạng thái. Đỉnh là vector còn cần mua; mỗi cạnh là "mua một món lẻ" hoặc "dùng một gói" với chi phí tương ứng; tìm đường ngắn nhất từ `needs` về vector `0`. Cách này không dùng công thức `f(cur)` hay thứ tự duyệt như các cách trên.

In [23]:
def oracle_dijkstra(price, special, needs):
    n = len(price)
    edges = [(tuple(1 if k == j else 0 for k in range(n)), price[j]) for j in range(n)]
    edges += [(tuple(o[:n]), o[n]) for o in special if any(o[:n])]
    start = tuple(needs)
    dist = {start: 0}
    pq = [(0, start)]
    while pq:
        d, s = heapq.heappop(pq)
        if d > dist.get(s, float("inf")):
            continue
        if not any(s):
            return d
        for vec, cost in edges:
            if all(vec[j] <= s[j] for j in range(n)):
                t = tuple(s[j] - vec[j] for j in range(n))
                nd = d + cost
                if nd < dist.get(t, float("inf")):
                    dist[t] = nd
                    heapq.heappush(pq, (nd, t))
    return None

METHODS = {
    "1-vet-can": Solution1().shoppingOffers,
    "2-co-nho": Solution2().shoppingOffers,
    "3-quay-lui": Solution3().shoppingOffers,
    "4-dp": Solution4().shoppingOffers,
    "nop": Solution().shoppingOffers,
}

def check_all(price, special, needs, expected=None):
    exp = oracle_dijkstra(price, special, needs) if expected is None else expected
    for name, fn in METHODS.items():
        got = fn(price, [o[:] for o in special], list(needs))
        assert got == exp, (name, price, special, needs, got, exp)
    return exp

for (args, expected) in examples:
    assert check_all(*args, expected=expected) == oracle_dijkstra(*args) == expected
check_all([3, 4], [[1, 1, 5], [2, 0, 4]], [2, 2], expected=10)     # tính tay ở mục 19
print("ví dụ của đề và ví dụ tự đặt: OK")

ví dụ của đề và ví dụ tự đặt: OK


### 12.2. Edge case

- `needs` toàn `0` -> chi phí `0`.
- Chỉ một mặt hàng (`n = 1`), có gói và không có gói rẻ hơn.
- Gói đắt hơn mua lẻ (phải bỏ qua), gói giá `0` (miễn phí).
- Gói **vừa khít** toàn bộ `needs`; gói **lớn hơn** `needs` ở một món (không dùng được).
- Gói rỗng (mọi món `0`) - đề đảm bảo không có, nhưng code vẫn phải chịu được.
- Giá lẻ bằng `0`.

In [24]:
edge = [
    ([2, 5], [[3, 0, 5]], [0, 0]),                           # không cần mua gì
    ([4], [[1, 5]], [3]),                                    # gói đắt hơn lẻ -> mua lẻ 3*4
    ([4], [[2, 5]], [5]),                                    # 2 gói (10) + 1 lẻ (4)
    ([4], [[2, 5]], [1]),                                    # gói cần 2 > nhu cầu 1
    ([3, 3], [[1, 1, 0]], [2, 2]),                           # gói miễn phí
    ([2, 3], [[2, 3, 4]], [2, 3]),                           # gói vừa khít
    ([2, 3], [[2, 4, 1]], [2, 3]),                           # gói cần 4B > 3B -> không dùng
    ([5, 5], [[1, 0, 0], [0, 0, 0]], [2, 2]),                # gói giá 0 cho 1A và gói rỗng
    ([0, 0], [[1, 1, 7]], [3, 3]),                           # giá lẻ 0
    ([10, 10, 10, 10, 10, 10], [[1, 1, 1, 1, 1, 1, 30]], [10] * 6),   # 6 mặt hàng, nhu cầu lớn
    ([1, 1], [[3, 0, 2], [0, 3, 2]], [5, 5]),                # gói 3 món giá 2
]
for price, special, needs in edge:
    exp = check_all(price, special, needs)
    print(f"price={price} needs={needs} special={special} -> {exp}")
assert check_all([2, 5], [[3, 0, 5]], [0, 0]) == 0
assert check_all([4], [[1, 5]], [3]) == 12
assert check_all([4], [[2, 5]], [5]) == 14
assert check_all([4], [[2, 5]], [1]) == 4
assert check_all([3, 3], [[1, 1, 0]], [2, 2]) == 0
assert check_all([2, 3], [[2, 3, 4]], [2, 3]) == 4
assert check_all([2, 3], [[2, 4, 1]], [2, 3]) == 13
assert check_all([0, 0], [[1, 1, 7]], [3, 3]) == 0
assert check_all([10] * 6, [[1, 1, 1, 1, 1, 1, 30]], [10] * 6) == 300
assert check_all([1, 1], [[3, 0, 2], [0, 3, 2]], [5, 5]) == 8      # mỗi mặt hàng: gói 3 món giá 2 + 2 lẻ = 4
print("edge case: OK")

price=[2, 5] needs=[0, 0] special=[[3, 0, 5]] -> 0
price=[4] needs=[3] special=[[1, 5]] -> 12
price=[4] needs=[5] special=[[2, 5]] -> 14
price=[4] needs=[1] special=[[2, 5]] -> 4
price=[3, 3] needs=[2, 2] special=[[1, 1, 0]] -> 0
price=[2, 3] needs=[2, 3] special=[[2, 3, 4]] -> 4
price=[2, 3] needs=[2, 3] special=[[2, 4, 1]] -> 13
price=[5, 5] needs=[2, 2] special=[[1, 0, 0], [0, 0, 0]] -> 10
price=[0, 0] needs=[3, 3] special=[[1, 1, 7]] -> 0


price=[10, 10, 10, 10, 10, 10] needs=[10, 10, 10, 10, 10, 10] special=[[1, 1, 1, 1, 1, 1, 30]] -> 300
price=[1, 1] needs=[5, 5] special=[[3, 0, 2], [0, 3, 2]] -> 8


edge case: OK


### 12.3. Phản ví dụ: input làm lộ các lỗi quen thuộc

- **Mua dư thì rẻ hơn** nhưng không được phép: ví dụ 2 của đề (gói `9` đồng cho `2A,2B,1C` bị cấm).
- **Tham lam theo mức giảm lớn nhất** thất bại (mục 13): có một cặp input được tìm tự động ở đó.
- **Gói rỗng** gây đệ quy vô hạn nếu không lọc.

In [25]:
tricky = [
    ([2, 3, 4], [[2, 2, 1, 9]], [1, 2, 1]),                  # chỉ có gói "mua dư": phải mua lẻ = 2+6+4
    ([1, 1], [[1, 1, 1], [2, 0, 1]], [2, 2]),                # nhiều đường tới cùng trạng thái
    ([5, 1], [[1, 0, 4], [0, 1, 0]], [3, 4]),                # gói miễn phí 1B
    ([3, 5], [[2, 1, 7], [1, 2, 8]], [3, 3]),                # hai gói chồng lấn, không chia hết
    ([2, 2], [[3, 0, 5], [0, 3, 5]], [4, 4]),                # gói 3 món giá 5 rẻ hơn lẻ 6, nhưng cần 4 món mỗi loại
]
for price, special, needs in tricky:
    print(price, special, needs, "->", check_all(price, special, needs))

[2, 3, 4] [[2, 2, 1, 9]] [1, 2, 1] -> 12
[1, 1] [[1, 1, 1], [2, 0, 1]] [2, 2] -> 2
[5, 1] [[1, 0, 4], [0, 1, 0]] [3, 4] -> 12
[3, 5] [[2, 1, 7], [1, 2, 8]] [3, 3] -> 15
[2, 2] [[3, 0, 5], [0, 3, 5]] [4, 4] -> 14


### 12.4. Đối chiếu ngẫu nhiên (seed cố định) với oracle Dijkstra

Sinh input theo ràng buộc thu nhỏ để cách vét cạn và quay lui còn chạy nhanh: `n` từ 1 đến 4, `needs[i]` từ 0 đến 5, giá lẻ từ 0 đến 10, từ 1 đến 8 gói với số món 0..4 và giá 0..30. Cho phép cả gói rỗng để kiểm tra độ bền.

In [26]:
rng = random.Random(638)
cnt = 0
for _ in range(500):
    n = rng.randint(1, 4)
    price = [rng.randint(0, 10) for _ in range(n)]
    needs = [rng.randint(0, 5) for _ in range(n)]
    special = []
    for _ in range(rng.randint(1, 8)):
        items = [rng.randint(0, 4) for _ in range(n)]
        special.append(items + [rng.randint(0, 30)])
    check_all(price, special, needs)
    cnt += 1
print("đã đối chiếu", cnt, "input ngẫu nhiên với", len(METHODS), "cách: OK")

đã đối chiếu 500 input ngẫu nhiên với 5 cách: OK


In [27]:
# Lượt thứ hai: gói "có lợi" (giá thấp hơn mua lẻ), n lên tới 5, đối chiếu 3 cách nhanh với oracle
rng = random.Random(6380)
FAST = ["2-co-nho", "3-quay-lui", "4-dp", "nop"]
for _ in range(150):
    n = rng.randint(1, 5)
    price = [rng.randint(1, 10) for _ in range(n)]
    needs = [rng.randint(0, 4) for _ in range(n)]
    special = []
    for _ in range(rng.randint(1, 10)):
        items = [rng.randint(0, 3) for _ in range(n)]
        if not any(items):
            items[rng.randrange(n)] = 1
        single = sum(p * c for p, c in zip(price, items))
        special.append(items + [max(0, single - rng.randint(1, 6))])
    exp = oracle_dijkstra(price, special, needs)
    for name in FAST:
        assert METHODS[name](price, [o[:] for o in special], list(needs)) == exp, (name, price, special, needs)
print("lượt 2: OK")

lượt 2: OK


### 12.5. Hiệu năng gần cận trên

Thử `n = 6` mặt hàng, `needs = [5,5,5,5,5,5]` (`6^6 = 46.656` trạng thái), 100 gói có lợi; cách 2 và cách 4 phải chạy trong chốc lát. Cách 1 (không nhớ) **không** được thử ở đây vì sẽ không bao giờ kết thúc.

In [28]:
rng = random.Random(7)
n = 6
price = [rng.randint(1, 10) for _ in range(n)]
special = []
for _ in range(100):
    items = [rng.randint(0, 3) for _ in range(n)]
    if not any(items):
        items[0] = 1
    single = sum(p * c for p, c in zip(price, items))
    special.append(items + [max(0, single - rng.randint(0, 5))])
needs = [5] * n

results = {}
for name in ["2-co-nho", "4-dp", "nop"]:
    t0 = time.perf_counter()
    results[name] = METHODS[name](price, special, needs)
    print(f"{name:10} -> {results[name]} trong {(time.perf_counter() - t0) * 1000:8.1f} ms")
assert len(set(results.values())) == 1
t0 = time.perf_counter()
print("oracle Dijkstra ->", oracle_dijkstra(price, special, needs), f"trong {(time.perf_counter() - t0) * 1000:.0f} ms")
assert oracle_dijkstra(price, special, needs) == results["nop"]

2-co-nho   -> 115 trong    332.6 ms


4-dp       -> 115 trong   1634.3 ms


nop        -> 115 trong    319.3 ms


oracle Dijkstra -> 115 trong 3881 ms


## 13. Các lỗi người mới hay gặp

### Lỗi 1: Cho phép mua dư (bỏ kiểm tra gói "vừa nhu cầu")

**Code sai:** chỉ trừ gói rồi cho `max(0, ...)`.

In [29]:
def wrong_allow_excess(price, special, needs):
    n = len(price)
    @lru_cache(None)
    def dfs(cur):
        best = sum(price[j] * cur[j] for j in range(n))
        for o in special:
            if any(o[:n]):
                nxt = tuple(max(0, cur[j] - o[j]) for j in range(n))     # SAI: cho phép mua dư
                if nxt != cur:
                    best = min(best, o[n] + dfs(nxt))
        return best
    return dfs(tuple(needs))

(args, expected) = examples[1]
print("sai  :", wrong_allow_excess(*args), "(dùng gói 9 đồng mua dư 1A)")
print("đúng :", Solution().shoppingOffers(*args))

sai  : 9 (dùng gói 9 đồng mua dư 1A)
đúng : 11


**Vì sao sai:** đề cấm mua dư. Gói `[2,2,1]` có `2A` trong khi nhu cầu chỉ `1A`; dùng nó là mua dư. Phải kiểm tra `offer[j] <= cur[j]` với **mọi** `j`.

**Code đúng:** `if all(o[j] <= cur[j] for j in range(n)):`.

### Lỗi 2: Tham lam theo "gói giảm giá nhiều nhất"

Ý nghĩ tự nhiên: luôn dùng gói tiết kiệm được nhiều tiền nhất (giá mua lẻ - giá gói) cho đến khi không còn gói vừa. Ta để máy **tìm** một phản ví dụ:

In [30]:
def wrong_greedy(price, special, needs):
    n = len(price)
    cur = list(needs)
    total = 0
    while True:
        best, best_save = None, 0
        for o in special:
            if any(o[:n]) and all(o[j] <= cur[j] for j in range(n)):
                save = sum(price[j] * o[j] for j in range(n)) - o[n]
                if save > best_save:
                    best, best_save = o, save
        if best is None:
            break
        total += best[n]
        cur = [cur[j] - best[j] for j in range(n)]
    return total + sum(price[j] * cur[j] for j in range(n))

rng = random.Random(2)
found = None
for _ in range(20000):
    n = rng.randint(1, 3)
    price = [rng.randint(1, 6) for _ in range(n)]
    needs = [rng.randint(1, 4) for _ in range(n)]
    special = []
    for _ in range(rng.randint(2, 3)):
        items = [rng.randint(0, 3) for _ in range(n)]
        if not any(items):
            items[0] = 1
        single = sum(p * c for p, c in zip(price, items))
        special.append(items + [max(0, single - rng.randint(1, 4))])
    if wrong_greedy(price, special, needs) != Solution().shoppingOffers(price, special, needs):
        found = (price, special, needs)
        break

price, special, needs = found
print("phản ví dụ:", "price =", price, "special =", special, "needs =", needs)
print("tham lam :", wrong_greedy(price, special, needs))
print("đúng     :", Solution().shoppingOffers(price, special, needs), "(oracle:", oracle_dijkstra(price, special, needs), ")")

phản ví dụ: price = [6, 4] special = [[1, 0, 3], [3, 2, 22], [3, 1, 20]] needs = [3, 4]
tham lam : 30
đúng     : 25 (oracle: 25 )


**Vì sao sai:** gói giảm nhiều nhất có thể "chiếm chỗ" những món mà hai gói nhỏ khác cùng dùng được hiệu quả hơn. Đây là bài **knapsack**: chọn cục bộ tốt nhất không đảm bảo toàn cục tốt nhất.

**Code đúng:** thử **mọi** lựa chọn, ghi nhớ (cách 2).

### Lỗi 3: Quên lựa chọn "mua lẻ phần còn lại"

In [31]:
def wrong_no_single(price, special, needs):
    n = len(price)
    INF = float("inf")
    @lru_cache(None)
    def dfs(cur):
        if not any(cur):
            return 0
        best = INF                                     # SAI: thiếu 'mua lẻ hết cur'
        for o in special:
            if any(o[:n]) and all(o[j] <= cur[j] for j in range(n)):
                best = min(best, o[n] + dfs(tuple(cur[j] - o[j] for j in range(n))))
        return best
    return dfs(tuple(needs))

(args, expected) = examples[0]
print("sai  :", wrong_no_single(*args), "(không có gói nào chia hết đúng 3A,2B)")
print("đúng :", Solution().shoppingOffers(*args))

sai  : inf (không có gói nào chia hết đúng 3A,2B)
đúng : 14


**Vì sao sai:** mua toàn bằng gói thường không thể khớp **chính xác** nhu cầu; phần dư phải mua lẻ. Mỗi trạng thái phải có phương án nền `best = sum(price[j] * cur[j])`.

**Code đúng:** khởi tạo `best` bằng giá mua lẻ hết `cur`, rồi so với các gói.

### Lỗi 4: Gói rỗng gây đệ quy vô hạn

In [32]:
def wrong_zero_offer(price, special, needs):
    n = len(price)
    @lru_cache(None)
    def dfs(cur):
        best = sum(price[j] * cur[j] for j in range(n))
        for o in special:                              # SAI: không bỏ gói rỗng
            if all(o[j] <= cur[j] for j in range(n)):
                best = min(best, o[n] + dfs(tuple(cur[j] - o[j] for j in range(n))))
        return best
    return dfs(tuple(needs))

old = sys.getrecursionlimit()
sys.setrecursionlimit(300)
try:
    wrong_zero_offer([2, 5], [[0, 0, 1], [3, 0, 5]], [3, 2])
except RecursionError:
    print("sai  : RecursionError (dfs(cur) gọi lại chính dfs(cur) mãi mãi)")
sys.setrecursionlimit(old)
print("đúng :", Solution().shoppingOffers([2, 5], [[0, 0, 1], [3, 0, 5]], [3, 2]))

sai  : RecursionError (dfs(cur) gọi lại chính dfs(cur) mãi mãi)
đúng : 15


**Vì sao sai:** gói `[0, 0, 1]` trừ đi `0` món nên `cur` **không đổi**; hàm gọi lại chính nó với cùng đối số -> vô hạn. Đề đảm bảo không có gói rỗng, nhưng thói quen tốt là **bỏ** chúng (`any(o[:n])`) hoặc luôn yêu cầu mỗi bước phải **giảm** `cur`.

**Code đúng:** lọc `any(o[:n])`.

### Lỗi 5: Dùng `list` làm khóa ghi nhớ

In [33]:
def wrong_list_key(price, special, needs):
    n = len(price)
    @lru_cache(None)
    def dfs(cur):                                      # cur là list -> không hashable
        best = sum(price[j] * cur[j] for j in range(n))
        for o in special:
            if any(o[:n]) and all(o[j] <= cur[j] for j in range(n)):
                best = min(best, o[n] + dfs([cur[j] - o[j] for j in range(n)]))
        return best
    return dfs(list(needs))

try:
    wrong_list_key(*examples[0][0])
except TypeError as e:
    print("sai  : TypeError:", e)
print("đúng :", Solution().shoppingOffers(*examples[0][0]))

sai  : TypeError: unhashable type: 'list'
đúng : 14


**Vì sao sai:** `lru_cache` dùng đối số làm khóa của một từ điển; `list` có thể thay đổi nên không hashable.

**Code đúng:** truyền `tuple` (`dfs(tuple(needs))` và `dfs(tuple(...))` ở lời gọi con).

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- **Nhận dạng:** "chọn các gói/đồng xu **dùng nhiều lần** để đạt **đúng** một mục tiêu với chi phí nhỏ nhất" = **unbounded knapsack**. Mục tiêu là **vector** (nhiều chiều) nên trạng thái là một tuple.
- **Giới hạn nhỏ** (`n <= 6`, `needs <= 10`) là tín hiệu: trạng thái là **tuple** của các số nhỏ -> ghi nhớ theo tuple, hoặc nén thành một số nguyên cơ số 11 (bitmask kiểu cơ số).
- Mẫu cố định: `f(cur) = min( mua lẻ , min_g giá(g) + f(cur - g) )`; nhớ **điều kiện vừa** (`g <= cur`) để không mua dư.
- **Lọc** các lựa chọn dư thừa trước (gói không rẻ hơn lẻ, gói rỗng) để chạy nhanh hơn và tránh vô hạn.
- Muốn tránh đếm trùng thứ tự: duyệt gói theo **chỉ số không giảm** (cách 3).
- Kiểm tra bằng **đường đi ngắn nhất** (Dijkstra) trên đồ thị trạng thái: một cách nhìn độc lập tốt để viết oracle.
- Bài họ hàng: LeetCode 322 (Coin Change), 518 (Coin Change II), 279 (Perfect Squares), 1125 (Smallest Sufficient Team, bitmask), 698 (Partition to K Equal Sum Subsets), 473 (Matchsticks to Square).

## 15. Biến thể

### 15.1. Khôi phục phương án mua (in các gói đã dùng)

Thay vì chỉ trả về tiền, ta muốn biết **dùng gói nào bao nhiêu lần** và mua lẻ cái gì. Lưu "lựa chọn tốt nhất" cho mỗi trạng thái rồi lần theo.

In [34]:
def shopping_with_plan(price, special, needs):
    n = len(price)
    offers = [(k, o) for k, o in enumerate(special)
              if any(o[:n]) and o[n] < sum(price[j] * o[j] for j in range(n))]
    choice = {}

    @lru_cache(None)
    def dfs(cur):
        best = sum(price[j] * cur[j] for j in range(n))
        pick = None
        for k, o in offers:
            if all(o[j] <= cur[j] for j in range(n)):
                nxt = tuple(cur[j] - o[j] for j in range(n))
                val = o[n] + dfs(nxt)
                if val < best:
                    best, pick = val, k
        choice[cur] = pick
        return best

    cost = dfs(tuple(needs))
    used, cur = [], tuple(needs)
    while choice[cur] is not None:
        k = choice[cur]
        used.append(k)
        cur = tuple(cur[j] - special[k][j] for j in range(n))
    return cost, used, cur

cost, used, rest = shopping_with_plan([2, 5], [[3, 0, 5], [1, 2, 10]], [3, 2])
print("tổng tiền:", cost, "| gói đã dùng (chỉ số):", used, "| mua lẻ phần còn lại:", rest)
cost, used, rest = shopping_with_plan([2, 3], [[1, 0, 1], [0, 1, 2], [1, 1, 2], [2, 1, 3]], [5, 4])
print("tổng tiền:", cost, "| gói đã dùng:", used, "| mua lẻ phần còn lại:", rest)
assert cost == Solution().shoppingOffers([2, 3], [[1, 0, 1], [0, 1, 2], [1, 1, 2], [2, 1, 3]], [5, 4])

tổng tiền: 14 | gói đã dùng (chỉ số): [1] | mua lẻ phần còn lại: (2, 0)
tổng tiền: 9 | gói đã dùng: [0, 2, 2, 2, 2] | mua lẻ phần còn lại: (0, 0)


### 15.2. Cho phép mua dư (biến thể "được mua thừa")

Nếu bỏ ràng buộc "không mua dư", một gói lớn có thể rẻ hơn. Khi đó gói luôn dùng được, phần vượt nhu cầu bị cắt về `0`. Đây là **lỗi 1** ở mục 13 trở thành **đề mới**; chú ý phải tránh vòng lặp (trạng thái không đổi).

In [35]:
def shopping_allow_excess(price, special, needs):
    n = len(price)
    offers = [o for o in special if any(o[:n])]

    @lru_cache(None)
    def dfs(cur):
        best = sum(price[j] * cur[j] for j in range(n))
        for o in offers:
            nxt = tuple(max(0, cur[j] - o[j]) for j in range(n))
            if nxt != cur:                           # phải tiến bộ, tránh lặp vô hạn
                best = min(best, o[n] + dfs(nxt))
        return best
    return dfs(tuple(needs))

print(shopping_allow_excess([2, 3, 4], [[1, 1, 0, 4], [2, 2, 1, 9]], [1, 2, 1]), "(được mua dư)  vs",
      Solution().shoppingOffers([2, 3, 4], [[1, 1, 0, 4], [2, 2, 1, 9]], [1, 2, 1]), "(đề gốc)")

9 (được mua dư)  vs 11 (đề gốc)


### 15.3. Trường hợp một chiều = bài đổi tiền

Khi `n = 1` bài này chính là: cần mua đúng `N` món, có các gói `(k món, giá)` và giá lẻ. Đó là unbounded knapsack ở mục 3.8. Hai cách giải cho ra cùng kết quả:

In [36]:
rng = random.Random(153)
for _ in range(200):
    p = rng.randint(1, 8)
    packs = [(rng.randint(1, 5), rng.randint(1, 20)) for _ in range(rng.randint(1, 4))]
    N = rng.randint(0, 20)
    special = [[k, c] for k, c in packs]
    assert Solution().shoppingOffers([p], special, [N]) == min_cost_exact(N, packs, p)
print("n = 1 trùng với dp một chiều: OK")

n = 1 trùng với dp một chiều: OK


### 15.4. Mỗi gói chỉ dùng tối đa một lần (0/1 knapsack)

Chỉ đổi cách duyệt: thay vì `cur`, trạng thái là `(chỉ số gói đang xét, cur)`; mỗi gói hoặc bỏ hoặc lấy một lần (xem bài tập 5 ở mục 17).

## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Vét cạn đệ quy | thử mọi thứ tự dùng gói | hàm mũ | `O(S)` | chỉ để hiểu đề |
| 2. Đệ quy có nhớ | `f(cur)` theo vector còn cần | `O(P*m*n)` trạng thái đến được | `O(P*n)` | **nộp bài** |
| 3. Quay lui theo chỉ số | mỗi đa tập gói một lần + cắt tỉa | tùy số đa tập | `O(S)` | ít gói, không muốn bảng nhớ |
| 4. DP từ dưới lên | duyệt mọi `cur` tăng dần, mã hóa cơ số hỗn hợp | `O(P*m*n)` mọi trạng thái | `O(P)` | muốn tránh đệ quy, cận trên rõ ràng |

Kết luận: nộp **cách 2**. Cách 4 là bản "bảng" của cùng công thức; cách 3 cho góc nhìn tổ hợp (đa tập); cách 1 là điểm xuất phát để thấy cần ghi nhớ.

## 17. 5 bài tập vận dụng (dễ → khó)

Cell **TODO** chỉ gán `None` hoặc định nghĩa hàm rỗng, **không được gọi** khi chạy notebook. Đáp án ở mục 19.

**Bài 1 (đoán output).** Dự đoán đầu ra rồi chạy:

In [37]:
print(Solution().shoppingOffers([2, 5], [[3, 0, 5], [1, 2, 10]], [0, 0]))
print(Solution().shoppingOffers([4], [[2, 5]], [5]))
print(Solution().shoppingOffers([2, 3], [[1, 1, 3]], [2, 1]))

0
14
5


**Bài 2 (tính tay).** `price = [3,4]`, `special = [[1,1,5],[2,0,4]]`, `needs = [2,2]`. Hãy liệt kê các cách (mua lẻ, gói 1, gói 2, ...) và gán chi phí nhỏ nhất vào `ans2` (đang là `None`).

In [38]:
ans2 = None      # TODO: điền một số nguyên

**Bài 3 (điền TODO).** Viết hai hàm trợ giúp: `base_cost(price, needs)` = tiền mua lẻ hết `needs`; và `fits(offer, cur, n)` = `True` nếu gói `offer` (độ dài `n + 1`) **vừa** nhu cầu `cur` (mọi món `offer[j] <= cur[j]`, không xét giá).

In [39]:
def base_cost(price, needs):
    # TODO
    pass

def fits(offer, cur, n):
    # TODO
    pass

**Bài 4 (tự viết lại).** Không nhìn mục 7, tự viết `my_shopping(price, special, needs)` theo cách 2 (đệ quy có nhớ, có lọc gói).

In [40]:
def my_shopping(price, special, needs):
    # TODO
    pass

**Bài 5 (mini challenge).** (a) Viết `num_states(needs)` trả về số trạng thái `prod(needs[j] + 1)`. (b) Viết `shopping_each_once(price, special, needs)`: mỗi gói chỉ được dùng **tối đa một lần** (không mua dư). Gợi ý: duyệt từng gói theo chỉ số, hoặc lấy hoặc bỏ.

In [41]:
def num_states(needs):
    # TODO
    pass

def shopping_each_once(price, special, needs):
    # TODO
    pass

## 18. Gợi ý 3 mức cho bài tập

**Mức 1 (nhẹ):**
- Bài 1: dòng 1 không cần mua gì; dòng 2: gói `2 món giá 5` rẻ hơn `2 * 4 = 8`; dòng 3: gói `1A 1B giá 3`, lẻ `2 + 3 = 5`.
- Bài 2: mua lẻ `2*3 + 2*4 = 14`; thử gói 1 một hoặc hai lần, gói 2 một lần.
- Bài 3: dùng `sum` với `zip` cho `base_cost`; dùng `all` cho `fits`.
- Bài 4: nhớ `tuple`, `lru_cache`, `best` khởi tạo bằng mua lẻ.
- Bài 5: `num_states` là tích; `shopping_each_once`: mỗi gói hai lựa chọn.

**Mức 2 (vừa):**
- Bài 2: gói 1 hai lần dùng hết `[2,2]` giá `10`; gói 2 một lần còn `[0,2]` mua lẻ `8` nên `4 + 8 = 12`.
- Bài 3: `sum(p * c for p, c in zip(price, needs))`; `all(offer[j] <= cur[j] for j in range(n))`.
- Bài 4: `any(o[:n])` bỏ gói rỗng; `o[n] < sum(price[j] * o[j])` lọc gói vô dụng.
- Bài 5(b): `@lru_cache` trên `(i, cur)`; `best = dfs(i + 1, cur)` (bỏ gói `i`) và nếu vừa thì `min(best, giá + dfs(i + 1, cur - gói))`.

**Mức 3 (gần đáp án):**
- Bài 2: đáp án nhỏ nhất là `10` (hai lần gói 1).
- Bài 5(b): trường hợp cơ sở `i == len(special)` trả về `base_cost(price, cur)`.
- Bài 5(a): `from math import prod`, `prod(c + 1 for c in needs)`.

## 19. Đáp án tham khảo

Chạy kiểm tra bằng `assert`.

In [42]:
# Bài 1: 0, 14, 3 + ... (gói 1A1B giá 3 + 1A lẻ 2 = 5)
assert Solution().shoppingOffers([2, 5], [[3, 0, 5], [1, 2, 10]], [0, 0]) == 0
assert Solution().shoppingOffers([4], [[2, 5]], [5]) == 14
assert Solution().shoppingOffers([2, 3], [[1, 1, 3]], [2, 1]) == 5

# Bài 2: gói 1 hai lần = 10
ans2 = 10
assert ans2 == Solution().shoppingOffers([3, 4], [[1, 1, 5], [2, 0, 4]], [2, 2]) == oracle_dijkstra([3, 4], [[1, 1, 5], [2, 0, 4]], [2, 2])
print("Bài 1, 2 OK")

Bài 1, 2 OK


In [43]:
# Bài 3
def base_cost(price, needs):
    return sum(p * c for p, c in zip(price, needs))

def fits(offer, cur, n):
    return all(offer[j] <= cur[j] for j in range(n))

assert base_cost([2, 5], [3, 2]) == 16
assert base_cost([2, 5], [0, 0]) == 0
assert fits([3, 0, 5], [3, 2], 2) and not fits([4, 0, 1], [3, 2], 2)
assert fits([0, 0, 9], [0, 0], 2)
print("Bài 3 OK")

Bài 3 OK


In [44]:
# Bài 4
def my_shopping(price, special, needs):
    n = len(price)
    offers = [o for o in special if any(o[:n]) and o[n] < base_cost(price, o[:n])]

    @lru_cache(None)
    def dfs(cur):
        best = base_cost(price, cur)
        for o in offers:
            if fits(o, cur, n):
                best = min(best, o[n] + dfs(tuple(cur[j] - o[j] for j in range(n))))
        return best

    return dfs(tuple(needs))

rng = random.Random(4)
for _ in range(200):
    n = rng.randint(1, 4)
    price = [rng.randint(0, 9) for _ in range(n)]
    needs = [rng.randint(0, 5) for _ in range(n)]
    special = [[rng.randint(0, 3) for _ in range(n)] + [rng.randint(0, 20)] for _ in range(rng.randint(1, 6))]
    assert my_shopping(price, special, needs) == oracle_dijkstra(price, special, needs)
print("Bài 4 OK")

Bài 4 OK


In [45]:
# Bài 5
from math import prod

def num_states(needs):
    return prod(c + 1 for c in needs)

def shopping_each_once(price, special, needs):
    n = len(price)

    @lru_cache(None)
    def dfs(i, cur):
        if i == len(special):
            return base_cost(price, cur)
        best = dfs(i + 1, cur)                                  # bỏ gói i
        o = special[i]
        if any(o[:n]) and fits(o, cur, n):                      # lấy gói i đúng một lần
            best = min(best, o[n] + dfs(i + 1, tuple(cur[j] - o[j] for j in range(n))))
        return best

    return dfs(0, tuple(needs))

assert num_states([3, 2]) == 12 and num_states([0, 0]) == 1 and num_states([10] * 6) == 11 ** 6

def brute_once(price, special, needs):
    n = len(price)
    best = None
    for mask in product([0, 1], repeat=len(special)):
        cur, cost, ok = list(needs), 0, True
        for take, o in zip(mask, special):
            if take:
                if not any(o[:n]) or not fits(o, cur, n):
                    ok = False
                    break
                cur = [cur[j] - o[j] for j in range(n)]
                cost += o[n]
        if ok:
            total = cost + base_cost(price, cur)
            best = total if best is None else min(best, total)
    return best

# ví dụ cho thấy "mỗi gói một lần" khác "dùng nhiều lần"
assert shopping_each_once([4], [[2, 5]], [5]) == 5 + 3 * 4
assert Solution().shoppingOffers([4], [[2, 5]], [5]) == 14
rng = random.Random(5)
for _ in range(300):
    n = rng.randint(1, 3)
    price = [rng.randint(0, 9) for _ in range(n)]
    needs = [rng.randint(0, 4) for _ in range(n)]
    special = [[rng.randint(0, 3) for _ in range(n)] + [rng.randint(0, 15)] for _ in range(rng.randint(1, 5))]
    assert shopping_each_once(price, special, needs) == brute_once(price, special, needs)
print("Bài 5 OK")

Bài 5 OK


## 20. Template nhớ nhanh + Checklist trước khi submit

**Template (đệ quy có nhớ theo vector còn cần):**

```python
from functools import lru_cache

class Solution:
    def shoppingOffers(self, price, special, needs):
        n = len(price)
        offers = [o for o in special
                  if any(o[:n]) and o[n] < sum(price[j] * o[j] for j in range(n))]   # lọc

        @lru_cache(None)
        def dfs(cur):                                   # cur: tuple còn cần mua
            best = sum(price[j] * cur[j] for j in range(n))      # mua lẻ hết
            for o in offers:
                if all(o[j] <= cur[j] for j in range(n)):        # không mua dư
                    best = min(best, o[n] + dfs(tuple(cur[j] - o[j] for j in range(n))))
            return best

        return dfs(tuple(needs))
```

**Checklist trước khi submit:**

- [ ] Mỗi gói chỉ dùng khi **vừa** nhu cầu (`o[j] <= cur[j]` với **mọi** `j`)?
- [ ] Luôn có phương án **mua lẻ** phần còn lại làm cận ban đầu?
- [ ] Khóa ghi nhớ là **tuple**, không phải `list`?
- [ ] Đã bỏ **gói rỗng** và (tùy chọn) gói **không rẻ hơn** mua lẻ?
- [ ] Mỗi bước thật sự **giảm** `cur` (đệ quy chắc chắn dừng)?
- [ ] Đã thử: `needs` toàn `0`, một mặt hàng, gói đắt hơn lẻ, gói vừa khít, gói lớn hơn nhu cầu?
- [ ] Không dùng tham lam "gói giảm nhiều nhất"?